# RSNA Setup
Setup for the RSNA kaggle competition which includes initial sampling of some of the raw data and translating the reports from their native language and feature extraction to assist in training

In [2]:
import pandas as pd
from tqdm.notebook import tqdm

from fast_langdetect import detect

import requests
import ssl

tqdm.pandas(desc="RSNA Doctor Medical Report Translation")

In [3]:

# Tell Python's global HTTPS context to ignore SSL verification errors
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [4]:
def get_language_and_region(text: str) -> dict:
    result = detect(text, model="full")
    lang_tag = result[0]['lang']
    confidence = result[0]['score']
    return {'language': lang_tag,'confidence': confidence }


In [5]:
raw = pd.read_csv("data/train.csv")

In [11]:
def get_prompt(language_code, report_text, model: str = "gemma2:2b"):
    prompt = f"""You are a precision medical translator. 
Translate the following medical report from {language_code} into clear, professional medical English.

STRICT RULES:
- Provide ONLY the direct English translation.
- Do NOT include any introduction, notes, explanations, or markdown code blocks.
- Do NOT alter, omit, or assume any clinical facts.

INPUT REPORT ({language_code}):
\"\"\"
{report_text}
\"\"\""""

    response = requests.post("http://localhost:11434/api/generate", json={
        "model": model, # swap with llama3.2:3b to compare
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.1
        }
    })
    try:
        return response.json()["response"]
    except KeyError:
        return response
    

In [12]:
raw["Report_Language"] = raw.Report.progress_apply(get_language_and_region)
raw["Report_Lang_Code"] = raw.Report_Language.str.get("language")
raw["Report_Lang_Confidence"] = raw.Report_Language.str.get("confidence")

RSNA Doctor Medical Report Translation:   0%|          | 0/4407 [00:00<?, ?it/s]

In [8]:
raw.head()

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture,Report_Language,Report_Lang_Code,Report_Lang_Confidence
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'language': 'es', 'confidence': 0.79564309120...",es,0.795643
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'language': 'de', 'confidence': 0.65166682004...",de,0.651667
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'language': 'es', 'confidence': 0.79755920171...",es,0.797559
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'language': 'en', 'confidence': 0.91118329763...",en,0.911183
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'language': 'fr', 'confidence': 0.81666159629...",fr,0.816662


In [15]:
sample = raw.sample(10).copy()

In [23]:
def add_translation_to_models(dataframe, models = ["qwen2.5:3b", "llama3.2:3b", "gemma2:2b"]):
    for model in models:

        results = []

        # Initialize the notebook progress bar manually over row iterations
        with tqdm(total=len(dataframe), desc=f"Translating using {model}") as pbar:
            for index, row in dataframe.iterrows():
                
                # 1. Dynamically update the metric description on the right side of the bar
                # Truncate long reports so the UI text doesn't overflow your screen
                report_snippet = row['Report'][:20] + "..." if len(row['Report']) > 20 else row['Report']
                
                pbar.set_postfix({
                    "Row": index + 1,
                    "Lang": row['Report_Lang_Code'],
                    "Doc_Length": len(row['Report']),
                    "Current_Txt": report_snippet
                })
                
                # 2. Run your isolated pipeline step (e.g., your Stage 2 translation)
                try:
                    translated_text = get_prompt(row['Report_Lang_Code'], row['Report'], model)
                    results.append(translated_text)
                except Exception as e:
                    # Crucial for overnight runs: log errors instead of crashing the 4-hour loop
                    results.append(f"ERROR: {str(e)}")
                    
                # 3. Manually advance the progress bar by 1 step
                pbar.update(1)

        # Assign the collected array back to your DataFrame columns cleanly
        dataframe[f'English_Translation_{model}'] = results
    return dataframe

In [24]:
sample

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,...,Contusion,Fracture,Report_Language,Report_Lang_Code,Report_Lang_Confidence,English_Translation_qwen2.5:3b,English_Translation_llama3.2:b,English_Translation_gemma2:2b,English_Translation_llama3.2:3b,English_Translation_deepseek-r1:8b
1157,1.2.826.0.1.3680043.8.498.13288244567806030859...,Tεχνική: Η εξέταση έγινε µε ακολουθίες παλµών ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'el', 'confidence': 0.99719494581...",el,0.997195,Technical: The examination was performed with ...,<Response [404]>,Technical: The examination was performed using...,The patient underwent an examination with foll...,Technique: Imaging was performed using T1-weig...
842,1.2.826.0.1.3680043.8.498.12426381920136706157...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'en', 'confidence': 0.64509999752...",en,0.645100,Exam Type: MRI RIGHT KNEE WITHOUT CONTRAST\nEx...,<Response [404]>,MRI Knee Right Without Contrast\n\nExam Date a...,MRI of the right knee was performed on a 1.5 T...,Exam Type: MRI KNEE RIGHT WO CONTRAST \nExam ...
1081,1.2.826.0.1.3680043.8.498.13114693814781737495...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'es', 'confidence': 0.86954104900...",es,0.869541,MRI of the knee technique. Results: Internal m...,<Response [404]>,MRI of the knee. Results: Meniscal tear. Chang...,Magnetic Resonance Imaging of the Knee: Result...,Technique: MRI of the knee. Results: Medial me...
3573,1.2.826.0.1.3680043.8.498.77611073647079414772...,[DATE]: MR rechterknie Klinische inlichtingen/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'de', 'confidence': 0.55358493328...",de,0.553585,[DATE]: Right knee MRI clinical inquiry/radiol...,<Response [404]>,Right knee clinical information/radiological r...,Patient's right knee clinical information/radi...,[DATE]: Right Knee MR \nClinical/radiological...
5,1.2.826.0.1.3680043.8.498.10018552945042470316...,Antecedentes Clínicos:\nCondromalacia rotulian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'es', 'confidence': 0.75061613321...",es,0.750616,Clinical History:\nOsteoarthritic Changes.\n\n...,<Response [404]>,Clinical History: Patellar chondromalacia.\n\n...,Clinical History:\nChondromalacia of the rotul...,Antecedentes Clínicos:\nPatellofemoral Cartila...
3179,1.2.826.0.1.3680043.8.498.67902658411525109462...,"In the medial compartment, the meniscus is not...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'en', 'confidence': 0.88974922895...",en,0.889749,"In the medial compartment, the meniscus is int...",<Response [404]>,"In the medial compartment, the meniscus is int...","The medial compartment, lateral compartment, a...","In the medial compartment, the meniscus is not..."
1952,1.2.826.0.1.3680043.8.498.34040631818813472592...,Findings: No significant joint effusion. No i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'en', 'confidence': 0.70788019895...",en,0.707880,Findings: No significant joint effusion. No in...,<Response [404]>,Findings: No significant joint effusion. No in...,Findings: No significant joint effusion. No in...,**Findings:** No significant joint effusion is...
2895,1.2.826.0.1.3680043.8.498.59835677513721120155...,Regelrechte Artikulation im rechten Kniegelenk...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'de', 'confidence': 0.90675920248...",de,0.906759,Right knee joint with restricted range of moti...,<Response [404]>,Right knee joint articulation is present. Age...,"Normal knee joint articulation. Age-related, n...",Proper joint articulation of the right knee. A...
4083,1.2.826.0.1.3680043.8.498.91328985601265188961...,Technique: MRI of the knee. ACL normal. MCL no...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'en', 'confidence': 0.40761202573...",en,0.407612,"MR

In [25]:
raw = add_translation_to_models(raw)

Translating using qwen2.5:3b:   0%|          | 0/4407 [00:00<?, ?it/s]

Translating using llama3.2:3b:   0%|          | 0/4407 [00:00<?, ?it/s]

Translating using gemma2:2b:   0%|          | 0/4407 [00:00<?, ?it/s]

In [33]:
raw.columns

Index(['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus',
       'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
       'Synovitis', 'Baker's', 'Contusion', 'Fracture', 'Report_Language',
       'Report_Lang_Code', 'Report_Lang_Confidence',
       'English_Translation_qwen2.5:3b', 'English_Translation_llama3.2:3b',
       'English_Translation_gemma2:2b'],
      dtype='str')

In [26]:
raw.to_pickle("report_translations_data.pkl")

In [32]:
print(raw.iloc[4].Report)

CONSTATATIONS :

Fractures :
Aucune.

Alignement articulaire :
Normal.

Changements dégénératifs :
Chondropatie rétropatellaire modérée à sévère avec fissuration profonde impliquant les facettes médiale et latérale, au niveau de la facette médiale et œdème osseux réactionnel [grade 4].
Chondropatie légère des surfaces portantes des compartiments médial et latéral sans fissuration profonde.

Signal de la moelle osseuse :
Par ailleurs normal.

Ligament croisé antérieur : Normal.
Ligament croisé postérieur : Normal.
Ligament collatéral médial : Léger épaississement proximal compatible avec une ancienne entorse de bas grade.
Complexe du ligament collatéral latéral : normal.

Ménisque médial : Légers changements dégénératifs dégénérés sans signe de signe.
Ménisque latéral : Normal.

Tendons extenseurs : Normal.
Rétinacula patellaire : Normal.
Pes anserinus : Normal.

Coin postéro-latéral et postérieur : Normal.

Épanchement articulaire : Pas de liquide significatif.
Kyste de Baker : Aucun.


In [35]:
print(raw.iloc[4]["English_Translation_qwen2.5:3b"])

Conclusions :

Fractures :
None.

Alignment of joints :
Normal.

Degenerative changes :
Osteoarthritis retropatella moderate to severe with deep fissuring involving the medial and lateral facets, at the medial facet and reactive osseous edema [grade 4].
Moderate osteoarthritis of the bearing surfaces of the medial and lateral compartments without deep fissuring.

Bone marrow signal :
Normal.

Anterior cruciate ligament : Normal.
Posterior cruciate ligament : Normal.
Medial collateral ligament : Mild thickening at the proximal end compatible with a previous low-grade sprain.
Lateral collateral ligament complex : normal.

Medial meniscus : Mild degenerative changes without signs of injury.
Lateral meniscus : Normal.

Extensor tendons : Normal.
Patellar retinaculum : Normal.
Pes anserinus : Normal.

Postero-lateral and posterior : Normal.

Joint effusion :
No significant fluid present.

Baker cyst : None.

Other significant findings :
None.

Conclusion :
Moderate to severe osteoarthritis 

In [36]:
print(raw.iloc[4]['English_Translation_llama3.2:3b'])

Fractures: None.

Joint alignment: Normal.

Degenerative changes:
- Reticular patellar chondropathy moderate to severe with deep fissuring involving the medial and lateral facets, at the level of the medial facet and osteochondral reaction [grade 4].
- Mild degenerative changes on weight-bearing surfaces of the medial and lateral compartments without deep fissuring.

Spinal cord signal: Normal.

Anterior cruciate ligament: Normal.
Posterior cruciate ligament: Normal.
Medial collateral ligament: Slight proximal thickening compatible with an old low-grade sprain.
Lateral collateral ligament complex: Normal.

Meniscus medial: Mild degenerative changes without signs of tears.
Meniscus lateral: Normal.

Extensor tendons: Normal.
Patellar retinacula: Normal.
Pes anserinus: Normal.

Posterior-lateral and posterior compartments: Normal.

Articular effusion: No significant fluid present.
Baker's cyst: None.

Significant findings: None.

Conclusion:
- Reticular patellar chondropathy moderate to 

In [57]:
raw['lang_code'] = raw.Report_Language.astype(str).str[14:16]

In [58]:
languages = pd.read_csv("data/iso_639-1.csv")

In [65]:
lang_tbl = languages[["639-1", "name"]].rename({"639-1":"lang_code"}, axis=1)

In [66]:
lang_tbl.columns

Index(['lang_code', 'name'], dtype='str')

In [67]:
raw.head()

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,...,Contusion,Fracture,Report_Language,Report_Lang_Code,Report_Lang_Confidence,English_Translation_qwen2.5:3b,English_Translation_llama3.2:3b,English_Translation_gemma2:2b,Source_Language,lang_code
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'es', 'confidence': 0.79564309120...",es,0.795643,MRI of the knee technique. Results: Internal m...,MRI of the knee. Results: Internal meniscal te...,MRI of the knee. Results: Meniscal tear intern...,es,es
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'de', 'confidence': 0.65166682004...",de,0.651667,[DATE]: * Right Knee MRI 15ch AA Clinical Info...,Date: MR Knie Rechts 15ch AA Clinical Informat...,[Date]: MR Knee Right 15ch Clinical Informatio...,de,de
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'es', 'confidence': 0.79755920171...",es,0.797559,Findings:\nNo significant abnormalities in the...,No significant alterations in the bone marrow....,Findings:\nNo significant abnormalities of the...,es,es
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'en', 'confidence': 0.91118329763...",en,0.911183,"In the medial compartment, the meniscus is int...","The medial compartment, meniscus is intact wit...","In the medial compartment, the meniscus is int...",en,en
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"{'language': 'fr', 'confidence': 0.81666159629...",fr,0.816662,Conclusions :\n\nFractures :\nNone.\n\nAlignme...,Fractures: None.\n\nJoint alignment: Normal.\n...,No fractures.\n\nArticular alignment: Normal.\...,fr,fr


In [70]:
raw.columns

Index(['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus',
       'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
       'Synovitis', 'Baker's', 'Contusion', 'Fracture', 'Report_Language',
       'Report_Lang_Code', 'Report_Lang_Confidence',
       'English_Translation_qwen2.5:3b', 'English_Translation_llama3.2:3b',
       'English_Translation_gemma2:2b', 'Source_Language', 'lang_code'],
      dtype='str')

In [75]:
raw.merge(lang_tbl, how="left", on="lang_code").drop(["ACL","MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture", "Report_Language", "Source_Language", "lang_code", "Report_Lang_Code", "Report_Lang_Confidence"], axis=1).to_csv("rsna_report_translations.csv")